In [1]:
# Import packages
import os
import sys
import zipfile
import subprocess
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil
from tqdm.notebook import tqdm
import csv
import glob
from pathlib import Path

from Bio import SeqIO
from collections import Counter
from xgboost import XGBClassifier
from xgboost import plot_importance
from lightgbm import LGBMClassifier

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, log_loss

# Errors ignore
import warnings
warnings.filterwarnings('ignore')

In [ ]:


def parse_kmer_dump(txt_path):
    df = pd.read_csv(txt_path, sep='\s+', header=None, names=["kmer", "count"])
    return df.set_index("kmer").T  # Transpose to make k-mers columns

df_kmers = parse_kmer_dump("AAFNOT.txt")
df_kmers.head()


In [ ]:
!docker run hello-world

In [ ]:
!docker ps -a

In [ ]:
with open("FastqTrain/ID_AAFNOT.fastq", "r") as f:
    for _ in range(12):
        print(f.readline().strip())


In [ ]:
!docker pull dmolik/k-mer-counting-tools

In [ ]:
!docker run --rm dmolik/k-mer-counting-tools kmc --version

In [ ]:
def run_kmc_all(fastq_dir=".", db_dir="kmc_db", tmp_dir="kmc_tmp", txt_dir="kmc_txt", k=5):
    fastq_dir = Path(fastq_dir)
    db_dir = Path(db_dir)
    tmp_dir = Path(tmp_dir)
    txt_dir = Path(txt_dir)

    # Ensure all directories exist
    db_dir.mkdir(exist_ok=True)
    tmp_dir.mkdir(exist_ok=True)
    txt_dir.mkdir(exist_ok=True)

    # Find all .fastq files in working directory
    fastq_files = list(fastq_dir.glob("*.fastq"))
    if not fastq_files:
        print("❌ No FASTQ files found.")
        return

    for fastq_file in tqdm(fastq_files, desc="Extracting kmers.."):
        sample_id = fastq_file.stem
        db_output = db_dir / f"{sample_id}_k{k}"
        txt_output = txt_dir / f"{sample_id}_k{k}.txt"

        print(f"🔄 Processing {fastq_file.name}")

        try:
            # Run KMC
            subprocess.run([
                "./kmc", f"-k{k}",
                str(fastq_file),
                str(db_output),
                str(tmp_dir)
            ], check=True)

            # Dump KMC output to text
            subprocess.run([
                "./kmc_tools",
                "transform",
                str(db_output),
                "dump",
                str(txt_output)
            ], check=True)

            print(f"✅ Dumped to: {txt_output.name}")

        except subprocess.CalledProcessError as e:
            print(f"❌ Error processing {fastq_file.name}: {e}")
            continue

    print("🎉 All files processed and saved to kmc_db/ and kmc_txt/")



In [ ]:
#Extracting kmers for test fastq files

k_values = [4, 5, 6, 7, 8, 9]

for k in tqdm(k_values, desc="Extracting kmers ..."):
    db_dir = f"kmc_test_db_k{k}"
    tmp_dir = f"kmc_test_tmp_k{k}"
    txt_dir = f"kmc_test_txt_k{k}"
    run_kmc_all(txt_dir=txt_dir, db_dir=db_dir, tmp_dir=tmp_dir, k=k)

print(f"Files processed successfully for all {k}-mers")

In [ ]:
def convert_txt_to_kmer_csvs(txt_dir, k, output_prefix="kmer_features"):
    root_dir = Path(os.getcwd())
    txt_dir = root_dir / txt_dir
    txt_files = sorted(txt_dir.glob("*.txt"))

    if not txt_files:
        print(f"⚠️ No files found in {txt_dir}")
        return

    feature_rows = []
    for txt_file in txt_files:
        try:
            sample_id = txt_file.stem.split("_k")[0]
            df = pd.read_csv(txt_file, sep="\s+", header=None, names=["kmer", "count"])
            df_row = df.set_index("kmer").T
            df_row["filename"] = sample_id
            feature_rows.append(df_row)
        except Exception as e:
            print(f"❌ Error processing {txt_file.name}: {e}")

    if not feature_rows:
        print("⚠️ No feature rows were parsed. Exiting....")
        return

    df_k = pd.concat(feature_rows, ignore_index=True).fillna(0)
    df_k.set_index("filename", inplace=True)

    output_csv = f"{output_prefix}_k{k}.csv"
    df_k.to_csv(output_csv)
    print(f"✅ Saved: {output_csv} — shape {df_k.shape}")


In [ ]:
# COnverting train txt to csvs

k_values = [4, 5, 6, 7, 8, 9]

for k in tqdm(k_values, desc="Converting txt files..."):
    txt_folder = f"kmc_train_txt_k{k}"
    try:
        convert_txt_to_kmer_csvs(txt_dir=txt_folder, k=k, output_prefix="train_kmer_features")
    except Exception as e:
        print(f"❌ Failed to convert k={k}: {e}")


In [3]:
def convert_txt_to_kmer_csvs(txt_dir, k, output_prefix="kmer_features"):
    root_dir = Path(os.getcwd())
    txt_dir = root_dir / txt_dir
    txt_files = sorted(txt_dir.glob("*.txt"))

    if not txt_files:
        print(f"⚠️ No files found in {txt_dir}")
        return

    print("🔍 First pass: Collecting all k-mers...")
    all_kmers = set()
    for txt_file in tqdm(txt_files):
        try:
            df = pd.read_csv(txt_file, sep=r"\s+", header=None, names=["kmer", "count"])
            all_kmers.update(df["kmer"])
        except Exception as e:
            print(f"❌ Error reading {txt_file.name}: {e}")

    all_kmers = sorted(all_kmers)  # consistent order
    header = ["filename"] + all_kmers

    output_csv = f"{output_prefix}_k{k}.csv"
    print("✍️ Second pass: Writing rows to CSV...")

    with open(output_csv, mode="w", newline="") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=header)
        writer.writeheader()

        for txt_file in tqdm(txt_files, desc=f"Processing files for k={k}"):
            try:
                sample_id = txt_file.stem.split("_k")[0]
                df = pd.read_csv(txt_file, sep=r"\s+", header=None, names=["kmer", "count"])
                row_dict = dict.fromkeys(all_kmers, 0)  # init all to 0
                row_dict["filename"] = sample_id

                for _, row in df.iterrows():
                    row_dict[row["kmer"]] = row["count"]

                writer.writerow(row_dict)
            except Exception as e:
                print(f"❌ Error processing {txt_file.name}: {e}")

    print(f"✅ Saved: {output_csv}")


In [ ]:
# Read Train txt files 

k_values = [4, 5, 6, 7, 8, 9]

for k in tqdm(k_values, desc="Converting txt files..."):
    txt_folder = f"kmc_train_txt_k{k}"
    try:
        convert_txt_to_kmer_csvs(txt_dir=txt_folder, k=k, output_prefix="train_kmer_features")
    except Exception as e:
        print(f"❌ Failed to convert k={k}: {e}")


Converting txt files...:   0%|          | 0/1 [00:00<?, ?it/s]

🔍 First pass: Collecting all k-mers...


  0%|          | 0/2901 [00:00<?, ?it/s]

✍️ Second pass: Writing rows to CSV...


Processing files for k=9:   0%|          | 0/2901 [00:00<?, ?it/s]

✅ Saved: train_kmer_features_k9.csv


In [ ]:
# Make sure convert_txt_to_kmer_csvs is already defined and imported

k_values = [4, 5, 6, 7, 8, 9]

for k in tqdm(k_values, desc="Converting txt files..."):
    txt_folder = f"kmc_test_txt_k{k}"
    try:
        convert_txt_to_kmer_csvs(txt_dir=txt_folder, k=k, output_prefix="test_kmer_features")
    except Exception as e:
        print(f"❌ Failed to convert k={k}: {e}")